# VoxCPM2: Multilingual TTS, Voice Design & Cloning Inference Examples

In [ ]:
import json
import boto3

### Query endpoint

In [ ]:
endpoint_name = "!!!name!!!"  # Replace with your endpoint name
region = "!!!region!!!"  # Replace with your endpoint's AWS region


def query_endpoint(payload):
    client = boto3.client("runtime.sagemaker", region_name=region)
    response = client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(payload).encode("utf-8"),
    )
    result = json.loads(response["Body"].read())
    print(json.dumps(result, indent=2, default=str))
    return result

### Basic English Text-to-Speech

Synthesize studio-quality 48 kHz English speech from plain text. The response returns a base64-encoded WAV plus its sample rate. inference_timesteps controls the number of diffusion steps (higher = higher quality, slower); cfg_value is the classifier-free guidance strength.

In [ ]:
payload = {'inputs': 'VoxCPM2 brings multilingual support, creative voice design, and '
           'controllable voice cloning.',
 'parameters': {'inference_timesteps': 10, 'cfg_value': 2.0}}

In [ ]:
payload = {
    "inputs": "VoxCPM2 brings multilingual support, creative voice design, and controllable voice cloning.",
    "parameters": {"inference_timesteps": 10, "cfg_value": 2.0},
}
resp = query_endpoint(payload)
import base64
with open("english.wav", "wb") as f:
    f.write(base64.b64decode(resp["audio_base64"]))
print("sample_rate:", resp["sample_rate"], "num_samples:", resp["num_samples"])

### Voice Design from a Natural-Language Description

VoxCPM2 can invent a brand-new voice from a natural-language description alone — no reference audio needed. Put the description in parentheses at the START of the text, followed by the content to speak. Try attributes like gender, age, tone, emotion, and pace.

In [ ]:
payload = {'inputs': '(A young woman, gentle and sweet voice)Hello, welcome to VoxCPM2!',
 'parameters': {'inference_timesteps': 10, 'cfg_value': 2.0}}

In [ ]:
payload = {
    "inputs": "(A young woman, gentle and sweet voice)Hello, welcome to VoxCPM2!",
    "parameters": {"inference_timesteps": 10, "cfg_value": 2.0},
}
resp = query_endpoint(payload)
import base64
with open("voice_design.wav", "wb") as f:
    f.write(base64.b64decode(resp["audio_base64"]))
print("Generated designed-voice audio at", resp["sample_rate"], "Hz")

### Multilingual Synthesis (no language tag)

VoxCPM2 supports 30 languages and infers the language directly from the input text — no explicit language tag is required. Here we synthesize Chinese.

In [ ]:
payload = {'inputs': '你好，欢迎使用 VoxCPM2 多语言语音合成。',
 'parameters': {'inference_timesteps': 10, 'cfg_value': 2.0}}

In [ ]:
payload = {
    "inputs": "你好，欢迎使用 VoxCPM2 多语言语音合成。",
    "parameters": {"inference_timesteps": 10, "cfg_value": 2.0},
}
resp = query_endpoint(payload)
import base64
with open("chinese.wav", "wb") as f:
    f.write(base64.b64decode(resp["audio_base64"]))
print("Chinese audio bytes:", len(base64.b64decode(resp["audio_base64"])))

### Expressive / Style-Controlled Speech

A parenthetical style cue can steer emotion and pace. Combined with a higher cfg_value, this produces more expressive delivery. Here we ask for a cheerful, slightly faster tone.

In [ ]:
payload = {'inputs': '(slightly faster, cheerful tone)Great news — your deployment is '
           'live and ready to go!',
 'parameters': {'inference_timesteps': 12, 'cfg_value': 2.5}}

In [ ]:
payload = {
    "inputs": "(slightly faster, cheerful tone)Great news — your deployment is live and ready to go!",
    "parameters": {"inference_timesteps": 12, "cfg_value": 2.5},
}
resp = query_endpoint(payload)
import base64
with open("expressive.wav", "wb") as f:
    f.write(base64.b64decode(resp["audio_base64"]))
print("Expressive audio at", resp["sample_rate"], "Hz")

### Fast Draft vs. High-Quality Rendering

Trade off speed and quality via inference_timesteps. Fewer steps (e.g. 4) give a quick draft; more steps (e.g. 16) give higher fidelity. This section demonstrates a low-step fast render for quick previews.

In [ ]:
payload = {'inputs': 'This is a quick low-step draft rendering of the requested text.',
 'parameters': {'inference_timesteps': 4, 'cfg_value': 2.0}}

In [ ]:
for steps in (4, 16):
    payload = {
        "inputs": "This is a quick low-step draft rendering of the requested text.",
        "parameters": {"inference_timesteps": steps, "cfg_value": 2.0},
    }
    resp = query_endpoint(payload)
    print(f"steps={steps}: num_samples={resp['num_samples']}")